In [1]:
import copy
import csv
from docplex.mp.model import Model
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import time

### Setup

In [2]:
NODE_TYPE = {
    0:'DEPOT',
    1:'PICKUP',
    2:'DESTINATION',
    3:'SINK'
}

class Node:
    def __init__(self,coor,task, node_type):
        self.coor = coor
        self.task_idx = task
        self.node_type = node_type
        
    def __repr__(self):
        return f" {NODE_TYPE[self.node_type]} {self.coor} of task {self.task_idx}"


class Task:
    def __init__(self,pickup_coor, dest_coor,robots_required:int,index):
        self.pickup_coor = pickup_coor
        self.dest_coor = dest_coor
        self.n = robots_required
        if index is not None:
            self.index = index
        else:
            self.index = -1
    
    def __repr__(self):
        return f"Task {self.index},{self.pickup_coor},{self.dest_coor},{self.n}"
    
    def expand_nodes(self) -> tuple:
        expanded_pickups = [Node(self.pickup_coor,self.index, 1) for _ in range(self.n)]
        expanded_dests = [Node(self.dest_coor,self.index, 2) for _ in range(self.n)]
        return (expanded_pickups, expanded_dests)
        pass

In [3]:
def gen_tasks(num_tasks,max_robots, coor_range = 500, no_delivery = False):
    """
    if no-delivery = true then every tasks has their pick up node = their destination node.
    """
    tasks = []
    for i in range(1,num_tasks + 1): # index start from 1 because 0 is the depot
        pickup_coor = rnd.uniform(0, coor_range,2)
        if no_delivery ==True:
            dest_coor = pickup_coor
        else:
            dest_coor = rnd.uniform(0, coor_range,2)
        if max_robots !=1:
            total_robots_required = rnd.randint(1,max_robots+1)
        else:
            total_robots_required = 1
        cur_tasks = Task(pickup_coor, dest_coor, total_robots_required, i)
        tasks.append(cur_tasks)
    
    return tasks

def gen_depot(coor_range = 500):
    coor = rnd.uniform(0, coor_range,2)
    depot = Node(coor, 0, 0)
    return depot

def construct_v_prime(depot,tasks):
    v_prime = [depot]
    for task in tasks:
        cur_pickups, cur_dests = task.expand_nodes()
        v_prime += cur_pickups
        v_prime += cur_dests
    
    return v_prime

def add_node_index(v_prime):
    
    total_nodes = len(v_prime)
    for i in range(total_nodes):
        v_prime[i].index = i

    return v_prime

def update_connection(v_prime):
    """
    Note that v_prime must have indicies already
    """
    depot = v_prime[0]
    depot.connection = [node.index for node in v_prime if node.index != depot.index and node.node_type == 1]# depot
    for cur_node in v_prime[1:]:
        cur_task_idx =  cur_node.task_idx
        cur_type = cur_node.node_type
        if cur_type == 1: #pickup
            cur_node.connection = [node.index for node in v_prime if node.task_idx == cur_task_idx and node.node_type ==2]
        elif cur_type == 2: #destination
            cur_node.connection = [node.index for node in v_prime if node.task_idx != cur_task_idx and node.node_type in [0,1]]
    return v_prime

def construct_e_prime(v_prime_connected):
    """
    Parameters
    ----------
    v_prime_connected: set of v_prime with info of connection
    """
    total_nodes = len(v_prime_connected)
    e_prime = {}
    e_prime ={
        i: v_prime_connected[i].connection for i in range(total_nodes)
    }
    return e_prime



def cal_dist(node1,node2):
    dist = np.linalg.norm(node1.coor - node2.coor)
    return dist

def create_cost_dict(connected_v_prime,node_idx_mapper):
    node_idx_mapper = {node.index: node for node in connected_v_prime}
    c = {}
    for cur_node in connected_v_prime:
        cur_idx = cur_node.index
        dist = {
            node_idx_mapper[node].index:cal_dist(cur_node,node_idx_mapper[node]) for node in cur_node.connection
        }
        c[cur_idx] = dist
    return c

def plot_map(depot, tasks):
    plt.scatter(depot.coor[0], depot.coor[1],c= 'C3', label = 'depot')
    for task in tasks:
        pickup_coor = task.pickup_coor
        dest_coor = task.dest_coor
        plt.scatter(pickup_coor[0], pickup_coor[1],c = 'C0',)
        plt.scatter(dest_coor[0], dest_coor[1], c ='C1',)
        xs = [pickup_coor[0],dest_coor[0]]
        ys = [pickup_coor[1],dest_coor[1]]
        plt.plot(xs, ys, c = PATH_COLOR[task.n])
        
    robots_required = set([task.n for task in tasks])
    legends_path ={f"{i} robot(s)":Line2D([0],[0], color = f"C{i}",lw = 2) for i in robots_required}
    legends_node = {
        "depot" : Line2D([0], [0], marker='o', color='w',markerfacecolor='C3', markersize= 8),
        "pickup" : Line2D([0], [0], marker='o', color='w',markerfacecolor='C0', markersize= 8),
        "dest." : Line2D([0], [0], marker='o', color='w',markerfacecolor='C1', markersize= 8),
    }
    legends = dict(legends_path,**legends_node)
    plt.legend(legends.values(), legends.keys())
    plt.plot()
    
def get_data_from_csv(path):
    df = pd.read_csv(path)
    data_dict = df.to_dict(orient = 'records')
    depot_raw = data_dict.pop(0)
    depot = Node(np.array([depot_raw['x_pickup'], depot_raw['y_pickup']]),0,0)
    for task in data_dict:
        task['pickup_coor'] = np.array([task['x_pickup'],task['y_pickup']])
        task['dest_coor'] = np.array([task['x_dest'],task['y_dest']])
        to_remove = ('x_pickup', 'y_pickup', 'x_dest', 'y_dest')
        [task.pop(field) for field in to_remove]
    tasks = [Task(**data) for data in data_dict]
    return depot, tasks

### Generate random variables

In [4]:
seed_num =  15
rnd = np.random
rnd.seed(seed_num)

In [5]:
T = 10 # total tasks
R = 5# total robots
no_delivery = False
gen_random_tasks = False
data_num = 4

data_path = f'data/data{data_num}.csv'
PATH_COLOR = {i: f'C{i}' for i in range(10)}
gap =0.1
time_limit = 1800
M = 10**6
indicator_methods = {
    1:"INDICATOR_CONSTRAINTS",
    2:"BIG_M_LAZY"
}
handle_indicator = 1

In [ ]:
if gen_random_tasks == True: 
    depot = gen_depot()
    tasks = gen_tasks(T,R, no_delivery = no_delivery)
else:
    depot,tasks = get_data_from_csv(data_path)

v_prime = construct_v_prime(depot,tasks)
add_node_index(v_prime)
v_prime = update_connection(v_prime)
node_idx_mapper = {node.index: node for node in v_prime}
c = create_cost_dict(v_prime,node_idx_mapper)
e_prime = construct_e_prime(v_prime)

In [ ]:
current_time = time.time()
plot_map(depot, tasks)
if gen_random_tasks == False:
    plt.savefig(f"img/{data_num}map.png", format="PNG")
else:
    plt.savefig(f"img/{seed_num}_{current_time}map_random.png", format="PNG")
plt.show()

## VRP Based Model

In [ ]:
def solve_model(tasks, 
                v_prime, 
                e_prime, 
                gap = 0.1, 
                time_limit = 3600, 
                log_output =True, 
                handle_indicator = 1):
    
    t_idx = [task.index for task in tasks]
    i_idx = [node.index for node in v_prime]
    ij_idx = [(i.index,j) for i in v_prime for j in i.connection]
    Pt = {task.index:[node.index for node in v_prime if (node.task_idx ==task.index and node.node_type ==1)] for task in tasks}
    Dt = {task.index:[node.index for node in v_prime if (node.task_idx ==task.index and node.node_type ==2)] for task in tasks}
    
    node_idx_mapper = {node.index: node for node in v_prime}
    c = create_cost_dict(v_prime,node_idx_mapper)

    model = Model("ThaoModel")
    x= model.binary_var_dict(ij_idx, name = 'x')
    y = model.continuous_var_dict(i_idx,lb =0, name = 'y')
    z = model.continuous_var_dict(i_idx,lb = 0, name = 'z')
    
    model.add_constraint(model.sum(x[0,j] for j in e_prime[0]) == R) #2
    model.add_constraint(model.sum(x[i,0] for i in e_prime if 0 in e_prime[i]) == R) #3
    model.add_constraints(model.sum(x[i,j] for i in e_prime if j in e_prime[i])==1 for j in i_idx[1:])
    model.add_constraints(model.sum(x[i,j] for j in e_prime[i])==1 for i in i_idx[1:])
    if handle_indicator ==1:
        indicator_SEC = [model.indicator_constraint(x[i,j],z[j] ==z[i]  +c[i][j] +y[j]) for i  in i_idx for j in e_prime[i] if j!=0]
        model.add_indicator_constraints(indicator_SEC)
        pass
    elif handle_indicator ==2:
        model.add_lazy_constraints(z[j] + M*(1-x[i,j]) >= z[i] +c[i][j] + y[j] for i  in i_idx for j in e_prime[i] if j!=0)
        model.add_lazy_constraints(z[j] - M*(1-x[i,j]) <= z[i] +c[i][j] + y[j] for i  in i_idx for j in e_prime[i] if j!=0)
    model.add_constraints(z[Pt[t][0]] == z[j] for t in t_idx for j in Pt[t])
    model.add_constraints(z[Dt[t][0]] == z[j] for t in t_idx for j in Dt[t])
    model.parameters.mip.tolerances.mipgap = gap
    model.time_limit = time_limit
    model.minimize(model.sum(c[i][j]*x[i,j] for i in i_idx for j in e_prime[i]) + model.sum(y[i.index] for i in v_prime))
    sol = model.solve(log_output = True)
    
    solved_gap = model.solve_details.mip_relative_gap
    solve_info = {
        'sol':sol,
        'x':x,
        'y':y,
        'gap':solved_gap
    }
    
    return solve_info

In [ ]:
Pt = {task.index:[node.index for node in v_prime if (node.task_idx ==task.index and node.node_type ==1)] for task in tasks}
Dt = {task.index:[node.index for node in v_prime if (node.task_idx ==task.index and node.node_type ==2)] for task in tasks}
t_idx = [task.index for task in tasks]
i_idx = [node.index for node in v_prime]
# ij_idx = [(i.index,j.index) for i in v_prime for j in v_prime]
ij_idx = [(i.index,j) for i in v_prime for j in i.connection]
model = Model("ThaoModel")
x= model.binary_var_dict(ij_idx, name = 'x')
y = model.continuous_var_dict(i_idx,lb =0, name = 'y')
z = model.continuous_var_dict(i_idx,lb = 0, name = 'z')

In [ ]:
model.add_constraint(model.sum(x[0,j] for j in e_prime[0]) == R) #2
model.add_constraint(model.sum(x[i,0] for i in e_prime if 0 in e_prime[i]) == R) #3
model.add_constraints(model.sum(x[i,j] for i in e_prime if j in e_prime[i])==1 for j in i_idx[1:])
model.add_constraints(model.sum(x[i,j] for j in e_prime[i])==1 for i in i_idx[1:])
if handle_indicator ==1:
    indicator_SEC = [model.indicator_constraint(x[i,j],z[j] ==z[i]  +c[i][j] +y[j]) for i  in i_idx for j in e_prime[i] if j!=0]
    model.add_indicator_constraints(indicator_SEC)
    pass
elif handle_indicator ==2:
    model.add_lazy_constraints(z[j] + M*(1-x[i,j]) >= z[i] +c[i][j] + y[j] for i  in i_idx for j in e_prime[i] if j!=0)
    model.add_lazy_constraints(z[j] - M*(1-x[i,j]) <= z[i] +c[i][j] + y[j] for i  in i_idx for j in e_prime[i] if j!=0)
model.add_constraints(z[Pt[t][0]] == z[j] for t in t_idx for j in Pt[t])
model.add_constraints(z[Dt[t][0]] == z[j] for t in t_idx for j in Dt[t])
model.parameters.mip.tolerances.mipgap = 0
model.time_limit = time_limit
model.minimize(model.sum(c[i][j]*x[i,j] for i in i_idx for j in e_prime[i]) + model.sum(y[i.index] for i in v_prime))


In [ ]:
print(model.export_as_lp())
sol = model.solve(log_output = True)

### Flow-based model

In [6]:
def construct_v(depot,tasks): # for flow model
    vs = [depot]
    for task in tasks:
        pickup_node = Node(task.pickup_coor,task.index,1)
        dest_node = Node(task.dest_coor,task.index,2)
        
        vs.append(pickup_node)
        vs.append(dest_node)
    sink = copy.deepcopy(depot)
    sink.task_idx = tasks[-1].index + 1
    sink.node_type = 3
    vs.append(sink)
    return vs

def cal_capacity(edge_jobs,edge_travels,node_idx_mapper,tasks):
    capac = {}
    for edge in edge_jobs:
        pu_node = edge[0]
        task_idx = node_idx_mapper[pu_node].task_idx
        cur_task = [task for task in tasks if task.index == task_idx][0]
        cur_capac = cur_task.n
        capac[edge] = cur_capac
    for edge in edge_travels:
        capac[edge] = 1
    return capac

def update_connection_flow(v_prime): #for flow model
    """
    Note that v_prime must have indicies already
    """
    depot = v_prime[0]
    depot.connection = [node.index for node in v_prime if (node.node_type == 1 or node.node_type ==3)]# depot
    for cur_node in v_prime[1:]:
        cur_task_idx =  cur_node.task_idx
        cur_type = cur_node.node_type
        if cur_type == 1: #pickup
            cur_node.connection = [node.index for node in v_prime if node.task_idx == cur_task_idx and node.node_type ==2]
        elif cur_type == 2: #destination
            cur_node.connection = [node.index for node in v_prime if node.task_idx != cur_task_idx and node.node_type in [1,3]]
    sink = v_prime[-1]
    sink.connection = []
    return v_prime

def construct_edge_travels(edges,node_idx_mapper): #for flow based model
    edge_travels = []
    for node in edges:
        if node_idx_mapper[node].node_type ==0: #depot
            edge_travels +=[(node,to_node) for to_node in edges[node]]
        if node_idx_mapper[node].node_type ==2: #destination
            edge_travels +=[(node,to_node) for to_node in edges[node]]
    return edge_travels

def construct_edge_jobs(edges,node_idx_mapper):  #for flow based model
    edge_jobs = []
    for node in edges:
        if node_idx_mapper[node].node_type ==1: #pickup
            edge_jobs +=[(node,to_node) for to_node in edges[node]]
    return edge_jobs


In [7]:
def solve_cplex_flow(tasks, 
                total_robots,
                nodes, 
                edges, 
                gap = 0.1, 
                time_limit = 3600, 
                log_output =True,
                export = True,
                handle_indicator = 1,
                ):
    
    j_idx = [task.index for task in tasks] #jobs
    u_idx = [node.index for node in nodes] #nodes
    sink_idx = len(nodes) -1 
    uv_idx = [(u.index,v) for u in nodes for v in u.connection] 
    M = total_robots
    
    node_idx_mapper = {node.index: node for node in nodes}
    c = create_cost_dict(nodes,node_idx_mapper)
    edge_travels = construct_edge_travels(es, node_idx_mapper)
    edge_jobs = construct_edge_jobs(es, node_idx_mapper)
    capac = cal_capacity(edge_jobs,edge_travels, node_idx_mapper,tasks)

    model = Model("ThaoFlowBasedModel")
    f= model.binary_var_dict(uv_idx, name = 'f')
    y = model.continuous_var_dict(uv_idx,lb =0, name = 'y')
    z = model.continuous_var_dict(u_idx,lb = 0, name = 'z')
    
    model.add_constraint(model.sum(f[0,v] for v in edges[0]) == total_robots) #14
    model.add_constraint(model.sum(f[u,sink_idx] for u in edges if sink_idx in edges[u]) == total_robots) #15
    model.add_constraints(model.sum(capac[(u,v)]*f[u,v] for u in u_idx if v in edges[u]) - model.sum(capac[(v,vp)]*f[v,vp] for vp in edges[v]) == 0 for v in u_idx[1:-1]) #16
#     model.add_constraints(f[u,v] <= 1 for (u,v) in edge_travels) #17 # ko cần if edge task
    model.add_constraints(f[u,v] == 1 for (u,v) in edge_jobs) #18
    model.add_constraint(z[0] == 0) #19
    model.add_constraints(z[v] == z[u] + c[u][v] for (u,v) in edge_jobs) #20
    if handle_indicator ==1:
        indicator_SEC = [model.indicator_constraint(f[u,v],z[v] ==z[u]  +c[u][v] +y[u,v]) for (u,v) in edge_travels] #21
        model.add_indicator_constraints(indicator_SEC) 
#     model.minimize(model.sum(c[u][v]*f[u,v] for (u,v) in edge_travels) +
#                    model.sum(c[u][v]*capac[(u,v)]*f[u,v] for (u,v) in edge_jobs) +
#                    model.sum(y[u,v] for u in u_idx for v in edges[u]))
    model.minimize(5)
    
    model.parameters.mip.tolerances.mipgap = gap
    model.time_limit = time_limit
    if export == True:
        print(model.export_as_lp())
    sol = model.solve(log_output = log_output)
    
    solved_gap = model.solve_details.mip_relative_gap
    solve_info = {
        'sol':sol,
        'x':f,
        'y':y,
        'gap':solved_gap
    }
    
    return solve_info

In [8]:
if gen_random_tasks == True: 
    depot = gen_depot()
    tasks = gen_tasks(T,R, no_delivery = no_delivery)
else:
    depot,tasks = get_data_from_csv(data_path)

In [9]:
total_robots = R
vs = construct_v(depot,tasks)
add_node_index(vs)
vs = update_connection_flow(vs)
node_idx_mapper = {node.index: node for node in vs}
c = create_cost_dict(vs,node_idx_mapper)
es = construct_e_prime(vs)

In [10]:
    sol_flow_result = solve_cplex_flow(tasks,total_robots,vs,es, gap =0)

C:\Users\CTNGUY~1\AppData\Local\Temp\ThaoFlowBasedModel.lp
Version identifier: 22.1.0.0 | 2022-03-09 | 1a383f8ce
CPXPARAM_Read_DataCheck                          1
CPXPARAM_TimeLimit                               3600
CPXPARAM_MIP_Tolerances_MIPGap                   0
Tried aggregator 2 times.
MIP Presolve eliminated 6 rows and 11 columns.
Aggregator did 5 substitutions.
Reduced MIP has 43 rows, 99 columns, and 180 nonzeros.
Reduced MIP has 31 binaries, 0 generals, 0 SOSs, and 31 indicators.
Presolve time = 0.02 sec. (0.15 ticks)
Probing time = 0.00 sec. (0.02 ticks)
Tried aggregator 1 time.
Detecting symmetries...
Reduced MIP has 43 rows, 99 columns, and 180 nonzeros.
Reduced MIP has 31 binaries, 0 generals, 0 SOSs, and 31 indicators.
Presolve time = 0.00 sec. (0.11 ticks)
Probing time = 0.00 sec. (0.02 ticks)
Clique table members: 2.
MIP emphasis: balance optimality and feasibility.
MIP search method: dynamic search.
Parallel mode: deterministic, using up to 6 threads.
Root relaxatio

In [11]:
sol = sol_flow_result['sol']
x = sol_flow_result['x']
sol

## Illustration

In [ ]:
def get_paths(moved_edges,total_robots):
    """
    Acquire the node sequence of each robot
    """
    left_edges = copy.deepcopy(moved_edges)
    paths = []
    for _ in range(total_robots):
        cur_path =[0]
        for edge in left_edges:
            if edge[0] ==0:
                cur_node = edge[1]
                cur_path.append(edge[1])
                left_edges.remove(edge)
                break

        while cur_node !=0:
            for edge in left_edges:
                if edge[0] == cur_node:
                    cur_node = edge[1]
                    cur_path.append(edge[1])
        paths.append(cur_path)
    
    return paths

def get_tasks_path(paths):
    """
    Acquire the task sequence for each robot
    """
    tasks_path = []
    for path in paths:
        task_path = [node_idx_mapper[node].task_idx for node in path]
        tasks_path.append(task_path)
    return tasks_path

def get_paths_coor(paths,node_idx_mapper):
    paths_coor = []
    for path in paths:
        xs = [node_idx_mapper[node].coor[0] for node in path]
        ys = [node_idx_mapper[node].coor[1] for node in path]
        cur_path_coors = [xs,ys]
        paths_coor.append(cur_path_coors)
    
    return paths_coor

def get_edge_paths(paths):
    edge_paths = []
    for path in paths:
        cur_edge_path = []
        for i in range(len(path)-1):
            cur_edge_path.append((path[i], path[i+1]))
        edge_paths.append(cur_edge_path)
    return edge_paths

In [ ]:
sol_dict = sol.get_value_dict(x)
moved_edges = [edge for edge in sol_dict if sol_dict[edge] ==1]


#____Correct flow model____
corrected_moved_edges = []
if vs[-1].node_type == 3: # depot
    sink_idx = len(vs) -1
    for edge in moved_edges:
        tmp_edge =list(edge)
        for i in range(len(tmp_edge)):
            if tmp_edge[i] == sink_idx:
                tmp_edge[i] = 0
        corrected_moved_edges.append(tmp_edge)

paths = get_paths(corrected_moved_edges,R)
tasks_path = get_tasks_path(paths)
paths_coor = get_paths_coor(paths,node_idx_mapper)
edge_paths = get_edge_paths(paths)

In [ ]:
pickup_xs= [task.pickup_coor[0] for task in tasks]
pickup_ys = [task.pickup_coor[1] for task in tasks]
dest_xs = [task.dest_coor[0] for task in tasks]
dest_ys = [task.dest_coor[1] for task in tasks]
plt.scatter(depot.coor[0], depot.coor[1],c= 'red')
plt.scatter(pickup_xs,pickup_ys, label = 'pickup')
plt.scatter(dest_xs,dest_ys, label = 'destination')
i=0
for path_coor in paths_coor:
    plt.plot(path_coor[0],path_coor[1],linestyle = (1,(3+i*2,5)), color = f"C{i}")
    i+=1
plt.legend()
plt.plot()

## Mass Solve

In [ ]:
class SolvedInstance():
    
    def __init__(self, seed, tasks, total_robots, v_prime, e_prime, sol_result, solve_time,instance_num):
        self.seed = seed
        self.tasks = tasks
        self.total_robots = total_robots
        self.num_tasks = len(tasks)
        self.num_nodes = len(v_prime)
        self.edges = e_prime
        self.solve_time = solve_time
        self.gap = self._get_gap(sol_result)
        self.instance_num = instance_num
        
    def _get_solve_time(self,sol_result):
        pass

    def _get_gap(self, solve_result):
        return solve_result['gap']

def mass_solve(T,R,seed =1, gap =0.1,time_limit = 1800, total_instances = 5, log_output = True):
    
    solved_instances = []
    for instance_num in range(total_instances):
        depot = gen_depot()
        tasks = gen_tasks(T,R)
        v_prime = construct_v_prime(depot, tasks)
        add_node_index(v_prime)
        v_prime = update_connection(v_prime)
        node_idx_mapper = {node.index: node for node in v_prime}
        c = create_cost_dict(v_prime,node_idx_mapper)
        e_prime = construct_e_prime(v_prime)
        try:
            tik = time.time()
            solved_result = solve_model(tasks, v_prime, e_prime,gap = gap, time_limit = time_limit, log_output = log_output)
            tok = time.time()
            solved_time = tok - tik
        except:
            print(f"There is problem solving {T} tasks, {R} robots,seed{seed}")
            continue
        try:
            solved_instance = SolvedInstance(seed, tasks,R, v_prime, e_prime,solved_result,solved_time, instance_num)
            solved_instances.append(solved_instance)
        except Exception as e:
            raise e
            continue
        
    return solved_instances

In [ ]:
Rs = [1,2,3,4]
Ts = [5,10,15]

In [ ]:
full_solved_instances = []
for T in Ts:
    for R in Rs:
        print(f'__________________________solving for {T} tasks and {R} robot__________________________')
        solved_instances = mass_solve(T,R)
        [full_solved_instances.append(solved_instance) for solved_instance in solved_instances]

In [ ]:
def results_to_df(full_solved_instances, compact = True):
    solved_dict = [solve_instance.__dict__ for solve_instance in full_solved_instances]
    df_solved = pd.DataFrame(solved_dict)
    if compact == True:
        df_solved.drop(['seed','tasks','edges','instance_num'],axis =1,inplace = True)
    return df_solved

In [ ]:
df_results = results_to_df(full_solved_instances)
df_results = df_results.round(1)

In [ ]:
print(df_results.to_latex())

In [ ]:
print(df_results.groupby(['total_robots','num_tasks']).mean().reset_index().round(2).to_latex())

### Visualize results

In [ ]:
import networkx as nx

In [ ]:
G = nx.DiGraph()
V =[node for node in node_idx_mapper.keys()]
E = moved_edges
pos = {idx: node.coor for idx,node in node_idx_mapper.items()}
        
G.add_nodes_from(V)        
G.add_edges_from(E)

color_map = []
for node in V:
    if node ==0:
        color_map.append('red')
    elif node_idx_mapper[node].node_type ==1:
        color_map.append('C0')
    elif node_idx_mapper[node].node_type ==2:
        color_map.append('C1')

In [ ]:
edges_color = []
edges_style = []
for edge in E:
    for i in range(len(edge_paths)):
        if edge in edge_paths[i]:
            edges_color.append(f'C{i}')
            edges_style.append((1,(3+i*2,5)))

In [ ]:
nx.draw(G,pos = pos, node_color = color_map,node_size =50, edge_color = edges_color, style = edges_style, arrowsize =15)
if gen_random_tasks == False:
    plt.savefig(f"img/{data_num}result.png", format="PNG")
else:
    plt.savefig(f"img/{seed_num}_{current_time}result_random.png", format="PNG")
plt.show()